# ICT-Greffe2 — Le quadruplet $(A, F, r, \pi)$ : rendre « élargir son espace » testable

**Navigation** : [Index](README.md) | [<< ICT-30 Inhibited Invention](ICT-30-InhibitedInvention-Python.ipynb) | [Version Python](ICT-Greffe2-EspaceAtteignable-Python.ipynb)

**Greffe 2 de la strate 7** (#13568). La strate 7 dit que les agents « élargissent leur espace ». La formule est nébuleuse — ce notebook la rend **mesurable** en nommant quatre objets, et en montrant que chacun des quatre peut bouger **indépendamment** des trois autres. Le banc consomme deux instruments déjà livrés par la série [Planners](../../SymbolicAI/Planners/README.md) :

- **[Planners-5c](../../SymbolicAI/Planners/02-Classical/Planners-05c-Differentiel-Atteignabilite-Python.ipynb)** — le *différentiel d'atteignabilité* : ce que l'ajout d'une primitive rend possible (bras **élargissement**) ;
- **[Planners-10b](../../SymbolicAI/Planners/04-NeuroSymbolic/Planners-10b-LLM-Space-Reducer-Python.ipynb)** — le LLM comme *réducteur d'espace* (bras **restriction**, moteur [aicpp](https://github.com/Julien-Livet/aicpp)).

Aucun des deux ne nomme le quadruplet complet ; aucun ne mesure le **noyau** (ce que l'ajout rend *inatteignable*). C'est ce que cette greffe ajoute — sur le **même domaine** que Planners-5c, en consommant son socle sans le re-dériver.


> **Statut épistémique** — **Sans verdict à ce jour** : aucune ligne de la [matrice de dissociations](../../../docs/ict/dissociations-matrix.md) ne concerne ce notebook ; son statut épistémique sera porté par la matrice le cas échéant.


## Le quadruplet, nommé

| Objet | Définition opérationnelle | Se mesure par |
|---|---|---|
| $A$ — espace ambiant | l'ensemble des états atteignables compte tenu du **vocabulaire** de primitives $V$ | BFS exhaustif : $A(V) = \{s : \exists \pi,\ run\ \pi\ s_0 \to s\}$ |
| $F$ — espace admissible | la part de $A$ que le **solveur** explore effectivement sous son budget $B$ (expansions) | BFS tronqué à $B$ expansions |
| $r$ — liberté résiduelle | le rapport $\lvert F_B\rvert / \lvert A\rvert$ : combien de l'ambiant reste jouable à budget donné | ratio direct |
| $\pi$ — politique | l'**ordre d'expansion** des successeurs (ici : déterministe, paramétrable) | coût en expansions de la première découverte d'un but |

Pourquoi **quatre** objets : dire « l'agent a progressé » sans dire *lequel* des quatre a bougé ne dit rien. Élargir $A$ sans vérifier $F$ peut **contracter** $r$ (l'espace croît plus vite que le budget) ; changer $\pi$ améliore la navigation sans toucher ni $A$ ni $F$. Les trois cas de figure de la strate 7 (contraction certifiée / navigation améliorée / élargissement réel) deviennent trois **signatures différentes** sur le quadruplet — et la greffe 1 de #13565 (Beck-Fiala) fournira le témoin de contraction *certifiée* quand il sera livré.


## Le socle STRIPS — copie de Planners-5c, garanties du lake

Le moteur est celui de Planners-5c (copie fidèle, non modifiée). Il consomme le lake `planning_lean` **sans le re-dériver** : les trois résultats certifiés sans `sorry` qui fondent la lecture sont cités en garantie dans la section suivante.


In [1]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Moteur STRIPS volontairement minimal (copie déclarée de Planners-5c) :
// un etat est un ensemble fini d'atomes, un operateur porte ses preconditions,
// sa liste d'ajouts et sa liste de suppressions.

public sealed class Etat
{
    private readonly HashSet<string> atomes;
    public string[] Atomes { get; }
    public string Key { get; }

    public Etat(IEnumerable<string> atomes)
    {
        // Canonicalisation : atomes dedupliques puis tries une bonne fois. Key est
        // independant de l'ordre d'insertion — c'est lui que la BFS compare pour
        // decider qu'un etat a deja ete visite (deux etats memes atomes = meme Key).
        var trie = atomes.Distinct(StringComparer.Ordinal).ToArray();
        Array.Sort(trie, StringComparer.Ordinal);
        Atomes = trie;
        this.atomes = new HashSet<string>(trie, StringComparer.Ordinal);
        Key = string.Join("|", trie);
    }

    public bool Contient(string atome) => atomes.Contains(atome);
    public bool Contient(IEnumerable<string> sousEnsemble) => sousEnsemble.All(atomes.Contains);
    // Plus/Moins appliquent les listes d'ajouts puis de suppressions de
    // l'operateur (semantique STRIPS : l'etat resultat reste un ensemble trie).
    public Etat Plus(IEnumerable<string> ajouts) => new Etat(Atomes.Concat(ajouts));
    public Etat Moins(IEnumerable<string> suppressions) => new Etat(AtomsHors(suppressions));
    private IEnumerable<string> AtomsHors(IEnumerable<string> suppressions) =>
        Atomes.Where(a => !suppressions.Contains(a, StringComparer.Ordinal));
    public override string ToString() => Key;
}

public sealed class Operateur
{
    public string Nom { get; }
    public IReadOnlySet<string> Precond { get; }
    public IReadOnlySet<string> Ajouts { get; }
    public IReadOnlySet<string> Dels { get; }

    public Operateur(string nom, IEnumerable<string> precond, IEnumerable<string> ajouts, IEnumerable<string> dels)
    {
        Nom = nom;
        Precond = new HashSet<string>(precond, StringComparer.Ordinal);
        Ajouts = new HashSet<string>(ajouts, StringComparer.Ordinal);
        Dels = new HashSet<string>(dels, StringComparer.Ordinal);
    }

    // Applicable quand toutes les preconditions atomiques sont dans l'etat.
    public bool Applicable(Etat etat) => Precond.All(etat.Contient);
    public override string ToString() => Nom;
}

// Cle de tri des operateurs : (rang, nom). Le rang laisse une famille
// d'operateurs passer avant une autre ; a rang egal l'ordre lexicographique du
// nom rend le developpement totalement deterministe (rejeu identique du banc).
public sealed class CleTri : IComparable<CleTri>
{
    public int Rang { get; }
    public string Nom { get; }
    public CleTri(int rang, string nom) { Rang = rang; Nom = nom; }
    public int CompareTo(CleTri autre)
    {
        int comparaison = Rang.CompareTo(autre.Rang);
        return comparaison != 0 ? comparaison : string.CompareOrdinal(Nom, autre.Nom);
    }
}

// Reussite = (nombre d'expansions jusqu'a la premiere decouverte, profondeur
// du but). null sur les deux champs = but hors de portee du vocabulaire.
public sealed class ResultatBut
{
    public int? Expansions { get; }
    public int? Distance { get; }
    public ResultatBut(int? expansions, int? distance) { Expansions = expansions; Distance = distance; }
    public string ExpansionsTexte => Expansions.HasValue ? Expansions.Value.ToString(CultureInfo.InvariantCulture) : "None";
    public string DistanceTexte => Distance.HasValue ? Distance.Value.ToString(CultureInfo.InvariantCulture) : "None";
}

public static class Strips
{
    // CleLex = politique par defaut : rang 0 partout, donc tri par nom seul.
    public static Func<Operateur, CleTri> CleLex { get; } = o => new CleTri(0, o.Nom);

    // Un pas de planification : les operateurs applicables a l'etat, appliques
    // dans l'ordre fixe par la cle de tri (chaque successeur est un nouvel Etat
    // canonique, l'etat courant n'est jamais mute).
    public static List<(Operateur Op, Etat Suivant)> Successeurs(
        Etat etat, IEnumerable<Operateur> ops, Func<Operateur, CleTri> cleTri = null)
    {
        cleTri ??= CleLex;
        var sortie = new List<(Operateur, Etat)>();
        foreach (var op in ops.OrderBy(o => cleTri(o)))
            if (op.Applicable(etat))
                sortie.Add((op, etat.Plus(op.Ajouts).Moins(op.Dels)));
        return sortie;
    }

    // BFS sur les Key canoniques : dist[k] = plus courte profondeur pour
    // atteindre k. budget borne le nombre d'expansions (null = explorer tout
    // l'espace atteignable) ; etats est rempli au passage pour que l'appelant
    // puisse relire les etats eux-memes, pas seulement leurs distances.
    public static Dictionary<string, int> BfsBudget(
        Etat initial,
        IEnumerable<Operateur> ops,
        int? budget,
        Dictionary<string, Etat> etats,
        Func<Operateur, CleTri> cleTri = null)
    {
        cleTri ??= CleLex;
        var dist = new Dictionary<string, int>(StringComparer.Ordinal) { [initial.Key] = 0 };
        etats[initial.Key] = initial;
        var file = new Queue<Etat>();
        file.Enqueue(initial);
        int expansions = 0;

        while (file.Count > 0 && (budget == null || expansions < budget.Value))
        {
            var s = file.Dequeue();
            expansions++;
            foreach (var (_, t) in Successeurs(s, ops, cleTri))
                if (!dist.ContainsKey(t.Key))
                {
                    dist[t.Key] = dist[s.Key] + 1;
                    etats[t.Key] = t;
                    file.Enqueue(t);
                }
        }
        return dist;
    }
}

// Controles : chaque verification imprime CONFORME/ECART et LEVE sur ecart —
// le carnet ne peut pas se terminer silencieusement sur une valeur fausse.
public static class Verifs
{
    public static void Entier(string nom, long calcule, long attendu)
    {
        bool ok = calcule == attendu;
        Console.WriteLine((ok ? "CONFORME" : "ECART") + " — " + nom + " : calcule = "
            + calcule.ToString(CultureInfo.InvariantCulture) + " | attendu = "
            + attendu.ToString(CultureInfo.InvariantCulture));
        if (!ok) throw new InvalidOperationException("Ecart sur le controle « " + nom + " »");
    }

    public static void Reel(string nom, double calcule, double attendu, double tolerance)
    {
        bool ok = Math.Abs(calcule - attendu) <= tolerance;
        Console.WriteLine((ok ? "CONFORME" : "ECART") + " — " + nom + " : calcule = "
            + calcule.ToString("R", CultureInfo.InvariantCulture) + " | attendu = "
            + attendu.ToString("R", CultureInfo.InvariantCulture));
        if (!ok) throw new InvalidOperationException("Ecart sur le controle « " + nom + " »");
    }

    public static void Exact(string nom, string calcule, string attendu)
    {
        bool ok = string.Equals(calcule, attendu, StringComparison.Ordinal);
        Console.WriteLine((ok ? "CONFORME" : "ECART") + " — " + nom + " : calcule = " + calcule + " | attendu = " + attendu);
        if (!ok) throw new InvalidOperationException("Ecart sur le controle « " + nom + " »");
    }

    public static void Vrai(string nom, bool condition)
    {
        Console.WriteLine((condition ? "CONFORME" : "ECART") + " — " + nom);
        if (!condition) throw new InvalidOperationException("Ecart sur le contrôle « " + nom + " »");
    }
}

Console.WriteLine("Moteur STRIPS chargé (copie Planners-5c) : step + BFS budgété + politique paramétrable.");

Moteur STRIPS chargé (copie Planners-5c) : step + BFS budgété + politique paramétrable.


### Ce que le lake garantit — et pourquoi il n'est pas re-dérivé ici

Le lake `planning_lean` (consommé par Planners-5b/5c) prouve, sans `sorry` :

| Résultat certifié | Lieu | Ce qu'il garantit ici |
|---|---|---|
| $run\ \pi\ s \subseteq runR\ \pi\ s$ (toute trajectoire réelle est une trajectoire relâchée) | `Planning/Relaxation.lean:56` | la relaxation ne peut pas « sauver » un but : si $g$ est inatteignable en relâché, il l'est réellement |
| $step \subseteq stepR$ | `Planning/Strips.lean:58` | le lemme central de la monotonie relâchée |
| admissibilité $h^+ \le h^*$ | `Planning/Admissibility.lean:41` | les minorants heuristiques ne mentent jamais sur la distance |

L'interdit de #13568 est explicite : **réimplémenter un espace d'états, une relaxation ou une heuristique dans `ict/` alors que `planning_lean` les prouve**. Ce notebook n'en réimplémente aucune : le BFS est l'*instrument de mesure* de $A$ et $F$ (c'est le rôle qu'il a déjà dans Planners-5c), les garanties sont citées, pas re-prouvées.


## Le domaine : deux rives, un gouffre, un radeau qui n'existe pas encore

Domaine de Planners-5c, copie fidèle : deux rives (colonnes 0-2 et 4-5), un gouffre (colonne 3, infranchissable), une planchette, une clé, un trésor. Trois buts : $g_1$ (balise sur la rive droite), $g_2$ (détenir le trésor), $g_3$ (détenir la clé). Dans le vocabulaire de base $A_t$, la rive droite est **inatteignable** — donc $g_1$ et $g_2$ aussi.


In [2]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Domaine jouet : une grille 2 rives x 3 rangs, separee par un gouffre en
// colonne 3. La rive gauche porte les colonnes 0-2, la droite les colonnes
// 4-5 ; la colonne 3 n'est JAMAIS enumeree — le gouffre n'est pas une case,
// donc aucun operateur move ne peut y mener ni en partir.

public sealed class CasePosition : IEquatable<CasePosition>
{
    public int Colonne { get; }
    public int Rang { get; }
    public CasePosition(int colonne, int rang) { Colonne = colonne; Rang = rang; }
    public bool Equals(CasePosition autre) => autre != null && Colonne == autre.Colonne && Rang == autre.Rang;
    public override bool Equals(object objet) => Equals(objet as CasePosition);
    public override int GetHashCode() => HashCode.Combine(Colonne, Rang);
}

public static class Domaine
{
    public static readonly int[] ColsGauche = { 0, 1, 2 };
    public static readonly int[] ColsDroite = { 4, 5 };
    public static readonly int[] Rangs = { 0, 1, 2 };

    // Les cases existantes sont le produit cartesien (gauche + droite) x rangs,
    // tries pour un affichage stable ; la colonne 3 est absente par construction.
    public static CasePosition[] Cases()
    {
        var cases = new List<CasePosition>();
        foreach (var c in ColsGauche.Concat(ColsDroite))
            foreach (var r in Rangs)
                cases.Add(new CasePosition(c, r));
        return cases.OrderBy(t => t.Colonne).ThenBy(t => t.Rang).ToArray();
    }

    public static string Prop(int c, int r) => $"at-x{c}-y{r}";
    // Etat initial : l'agent en (0,0), la planchette POSEE en (1,1) (il faudra
    // la ramasser pour traverser), la cle et le tresor au sol a ramasser.
    public static Etat Initial() => new Etat(new[] { Prop(0, 0), "plank-at-x1-y1", "key-at-x2-y0", "treasure-at-x4-y1" });

    public static List<Operateur> OpsDeBase()
    {
        var ops = new List<Operateur>();
        var directions = new (int dc, int dr, string nom)[] { (1, 0, "est"), (-1, 0, "ouest"), (0, 1, "nord"), (0, -1, "sud") };
        var cases = Cases();
        // Deplacements : un move par (case, direction) MAIS seulement si la case
        // cible existe — cases.Contains(cible) rejette les bords du plateau ET
        // tout saut au-dessus du gouffre : rive gauche et rive droite sont deux
        // composantes connexes disjointes sans operateur supplementaire.
        foreach (var position in cases)
            foreach (var direction in directions)
            {
                var cible = new CasePosition(position.Colonne + direction.dc, position.Rang + direction.dr);
                if (!cases.Contains(cible)) continue;
                ops.Add(new Operateur($"move-{direction.nom}-x{position.Colonne}-y{position.Rang}",
                    new[] { Prop(position.Colonne, position.Rang) },
                    new[] { Prop(cible.Colonne, cible.Rang) },
                    new[] { Prop(position.Colonne, position.Rang) }));
            }

        // Ramassages : exiger d'etre sur la case de l'objet (precondition de
        // co-localisation) et remplacer « objet-at-... » par « holding-objet ».
        foreach (var objet in new[] { ("plank", 1, 1), ("key", 2, 0), ("treasure", 4, 1) })
            ops.Add(new Operateur($"pickup-{objet.Item1}",
                new[] { Prop(objet.Item2, objet.Item3), $"{objet.Item1}-at-x{objet.Item2}-y{objet.Item3}" },
                new[] { $"holding-{objet.Item1}" },
                new[] { $"{objet.Item1}-at-x{objet.Item2}-y{objet.Item3}" }));
        return ops;
    }

    // g1 = atteindre la balise sur la rive DROITE (impossible sans traverser) ;
    // g2/g3 = detenir le tresor / la cle (le tresor, pose a droite, exige aussi
    // la traverse).
    public static Dictionary<string, string[]> Buts() => new Dictionary<string, string[]>(StringComparer.Ordinal)
    {
        ["g1: balise (5,2)"] = new[] { Prop(5, 2) },
        ["g2: detenir le tresor"] = new[] { "holding-treasure" },
        ["g3: detenir la cle"] = new[] { "holding-key" },
    };
}

var cases = Domaine.Cases();
var initial = Domaine.Initial();
var buts = Domaine.Buts();
var nomsButs = buts.Keys.OrderBy(k => k, StringComparer.Ordinal).ToArray();

Console.WriteLine($"Cases existantes : {cases.Length.ToString(CultureInfo.InvariantCulture)} (colonne 3 = gouffre)");
Console.WriteLine("Buts : [" + string.Join(", ", nomsButs.Select(k => "'" + k + "'")) + "]");

Verifs.Entier("cases = (COLS_GAUCHE + COLS_DROITE) x RANGS", cases.Length,
    (Domaine.ColsGauche.Length + Domaine.ColsDroite.Length) * Domaine.Rangs.Length);
Verifs.Entier("parite |CASES| avec la source", cases.Length, 15);
Verifs.Vrai("la colonne 3 (gouffre) est absente du domaine", cases.All(t => t.Colonne != 3));
Verifs.Entier("atomes de l'etat initial", initial.Atomes.Length, 4);
Verifs.Exact("etat initial : la planchette est posee en (1,1)",
    string.Join(",", initial.Atomes),
    string.Join(",", new[] { "at-x0-y0", "key-at-x2-y0", "plank-at-x1-y1", "treasure-at-x4-y1" }.OrderBy(a => a, StringComparer.Ordinal)));
Verifs.Entier("nombre de buts", buts.Count, 3);
Verifs.Vrai("g1 exige la balise en (5,2)", buts["g1: balise (5,2)"].Contains(Domaine.Prop(5, 2)));
Verifs.Vrai("g2 exige le tresor en main", buts["g2: detenir le tresor"].Contains("holding-treasure"));
Verifs.Vrai("g3 exige la cle en main", buts["g3: detenir la cle"].Contains("holding-key"));

Cases existantes : 15 (colonne 3 = gouffre)


Buts : ['g1: balise (5,2)', 'g2: detenir le tresor', 'g3: detenir la cle']


CONFORME — cases = (COLS_GAUCHE + COLS_DROITE) x RANGS : calcule = 15 | attendu = 15


CONFORME — parite |CASES| avec la source : calcule = 15 | attendu = 15


CONFORME — la colonne 3 (gouffre) est absente du domaine


CONFORME — atomes de l'etat initial : calcule = 4 | attendu = 4


CONFORME — etat initial : la planchette est posee en (1,1) : calcule = at-x0-y0,key-at-x2-y0,plank-at-x1-y1,treasure-at-x4-y1 | attendu = at-x0-y0,key-at-x2-y0,plank-at-x1-y1,treasure-at-x4-y1


CONFORME — nombre de buts : calcule = 3 | attendu = 3


CONFORME — g1 exige la balise en (5,2)


CONFORME — g2 exige le tresor en main


CONFORME — g3 exige la cle en main


In [3]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

public sealed class Vocabulaire
{
    public string Nom { get; }
    public List<Operateur> Ops { get; }
    public Vocabulaire(string nom, List<Operateur> ops) { Nom = nom; Ops = ops; }
}

public static class Vocabulaires
{
    // Le radeau : un operateur par rang, de la colonne 2 a la colonne 4 — le
    // seul pont au-dessus du gouffre. Il exige holding-plank ET le SUPPRIME :
    // la planchette se consomme au passage (le radeau coule), le retour n'est
    // donc pas possible dans le meme plan sans une seconde planchette.
    public static List<Operateur> OpsAvecRadeau()
    {
        var ops = Domaine.OpsDeBase();
        foreach (var r in Domaine.Rangs)
            ops.Add(new Operateur($"raft-across-y{r}",
                new[] { Domaine.Prop(2, r), "holding-plank" },
                new[] { Domaine.Prop(4, r) },
                new[] { Domaine.Prop(2, r), "holding-plank" }));
        return ops;
    }

    // Jeter la planchette : primitive STERILE (liste d'ajouts vide) — elle
    // n'ouvre aucun nouvel etat porteur, elle ne fait qu'encombrer l'espace.
    // C'est le predicat de l'experience : mesurer ce qu'un operateur inutile
    // ajoute a l'espace atteignable et au cout de decouverte.
    public static List<Operateur> OpsAvecJeter(IEnumerable<Operateur> baseOps)
    {
        var ops = new List<Operateur>(baseOps);
        ops.Add(new Operateur("discard-plank",
            new[] { "holding-plank" }, Array.Empty<string>(), new[] { "holding-plank" }));
        return ops;
    }

    // Grille 2x2 de l'experience : {base, +raft} x {sans, +discard} — quatre
    // vocabulaires qui isolent l'effet du radeau (ouvre g1/g2) et celui du
    // discard (n'ouvre rien, rallonge la recherche).
    public static List<Vocabulaire> Tous()
    {
        var aT = Domaine.OpsDeBase();
        var ster = OpsAvecJeter(aT);
        var aT1 = OpsAvecRadeau();
        var t1s = OpsAvecJeter(aT1);
        return new List<Vocabulaire>
        {
            new Vocabulaire("A_t", aT),
            new Vocabulaire("A_t+discard", ster),
            new Vocabulaire("A_t1 (+raft)", aT1),
            new Vocabulaire("A_t1+discard", t1s),
        };
    }
}

var vocabs = Vocabulaires.Tous();
foreach (var v in vocabs)
    Console.WriteLine(v.Nom.PadRight(16) + "|V| = " + v.Ops.Count.ToString(CultureInfo.InvariantCulture) + " operateurs");

Verifs.Entier("|A_t| (deplacements + pickups)", vocabs[0].Ops.Count,
    vocabs[0].Ops.Count(o => o.Nom.StartsWith("move-", StringComparison.Ordinal))
    + vocabs[0].Ops.Count(o => o.Nom.StartsWith("pickup-", StringComparison.Ordinal)));
Verifs.Entier("|A_t+discard| (+1 primitive sterile)", vocabs[1].Ops.Count, vocabs[0].Ops.Count + 1);
Verifs.Entier("|A_t1 (+raft)| (+3 raft)", vocabs[2].Ops.Count, vocabs[0].Ops.Count + 3);
Verifs.Entier("|A_t1+discard| (+3 raft, +1 discard)", vocabs[3].Ops.Count, vocabs[0].Ops.Count + 4);
Verifs.Vrai("discard est present dans A_t+discard et A_t1+discard",
    vocabs[1].Ops.Any(o => o.Nom == "discard-plank") && vocabs[3].Ops.Any(o => o.Nom == "discard-plank"));
Verifs.Vrai("discard est absent de A_t et A_t1",
    !vocabs[0].Ops.Any(o => o.Nom == "discard-plank") && !vocabs[2].Ops.Any(o => o.Nom == "discard-plank"));
Verifs.Entier("nombre de raft (un par rang)", vocabs[2].Ops.Count(o => o.Nom.StartsWith("raft", StringComparison.Ordinal)), 3);
Verifs.Vrai("raft exige holding-plank et supprime la planchette tenue",
    vocabs[2].Ops.Where(o => o.Nom.StartsWith("raft", StringComparison.Ordinal))
        .All(o => o.Precond.Contains("holding-plank") && o.Dels.Contains("holding-plank")));

A_t             |V| = 41 operateurs


A_t+discard     |V| = 42 operateurs


A_t1 (+raft)    |V| = 44 operateurs


A_t1+discard    |V| = 45 operateurs


CONFORME — |A_t| (deplacements + pickups) : calcule = 41 | attendu = 41


CONFORME — |A_t+discard| (+1 primitive sterile) : calcule = 42 | attendu = 42


... (2 outputs left out)

CONFORME — discard est present dans A_t+discard et A_t1+discard


CONFORME — discard est absent de A_t et A_t1


CONFORME — nombre de raft (un par rang) : calcule = 3 | attendu = 3


CONFORME — raft exige holding-plank et supprime la planchette tenue


## $A$ — le différentiel d'atteignabilité **système**

Première mesure : l'espace ambiant $A(V)$ en entier (BFS exhaustif), et les buts atteignables dans chacun.


In [4]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Tables globales de l'experience : pour chaque vocabulaire, l'espace atteignable
// complet (etats par Key) et sa table de distances — remplies une fois ici,
// relues par les cellules suivantes (deltas, BFS par but).

public static class Banc
{
    public static Dictionary<string, Dictionary<string, Etat>> Atteint { get; } =
        new Dictionary<string, Dictionary<string, Etat>>(StringComparer.Ordinal);
    public static Dictionary<string, Dictionary<string, int>> Dist { get; } =
        new Dictionary<string, Dictionary<string, int>>(StringComparer.Ordinal);
}

var vocabsA = Vocabulaires.Tous();
var butsA = Domaine.Buts();
var nomsButsA = butsA.Keys.OrderBy(k => k, StringComparer.Ordinal).ToArray();
// Valeurs ancrees de la source : tailles d'espace atteignable par vocabulaire,
// puis nombre de buts atteignables (confinement a 1/3 sans le raft, 3/3 avec).
var attendusA = new long[] { 36, 54, 60, 78 };
var attendusButsA = new long[] { 1, 1, 3, 3 };

for (int i = 0; i < vocabsA.Count; i++)
{
    var v = vocabsA[i];
    var etats = new Dictionary<string, Etat>(StringComparer.Ordinal);
    // BFS complete sans budget : dist.Count = |A|, la taille exacte de l'espace
    // atteignable — la mesure brute de ce que chaque extension ajoute.
    var dist = Strips.BfsBudget(Domaine.Initial(), v.Ops, null, etats);
    Banc.Dist[v.Nom] = dist;
    Banc.Atteint[v.Nom] = etats;
    // Un but est atteignable si AU MOINS UN etat atteint le satisfait.
    var butsOk = nomsButsA.Where(b => etats.Values.Any(s => s.Contient(butsA[b]))).ToArray();
    var court = butsOk.Select(b => "'" + b.Substring(0, 2) + "'").ToArray();
    Console.WriteLine(v.Nom.PadRight(16) + "|A| = " + dist.Count.ToString(CultureInfo.InvariantCulture).PadLeft(4)
        + " etats   buts atteignables : " + butsOk.Length.ToString(CultureInfo.InvariantCulture) + "/3 ["
        + string.Join(", ", court) + "]");
    Verifs.Entier("|A| de " + v.Nom, dist.Count, attendusA[i]);
    Verifs.Entier("buts atteignables de " + v.Nom, butsOk.Length, attendusButsA[i]);
}

Verifs.Vrai("A_t et A_t+discard n'atteignent que g3 (la cle)",
    Banc.Atteint["A_t"].Values.Any(s => s.Contient("holding-key"))
    && !Banc.Atteint["A_t"].Values.Any(s => s.Contient("holding-treasure"))
    && !Banc.Atteint["A_t"].Values.Any(s => s.Contient(Domaine.Prop(5, 2))));
Verifs.Vrai("A_t1 et A_t1+discard atteignent les trois buts",
    new[] { "A_t1 (+raft)", "A_t1+discard" }.All(n => nomsButsA.All(b =>
        Banc.Atteint[n].Values.Any(s => s.Contient(butsA[b])))));

A_t             |A| =   36 etats   buts atteignables : 1/3 ['g3']


CONFORME — |A| de A_t : calcule = 36 | attendu = 36


CONFORME — buts atteignables de A_t : calcule = 1 | attendu = 1


A_t+discard     |A| =   54 etats   buts atteignables : 1/3 ['g3']


CONFORME — |A| de A_t+discard : calcule = 54 | attendu = 54


CONFORME — buts atteignables de A_t+discard : calcule = 1 | attendu = 1


... (4 outputs left out)

CONFORME — |A| de A_t1+discard : calcule = 78 | attendu = 78


CONFORME — buts atteignables de A_t1+discard : calcule = 3 | attendu = 3


CONFORME — A_t et A_t+discard n'atteignent que g3 (la cle)


CONFORME — A_t1 et A_t1+discard atteignent les trois buts


### Lecture : le différentiel non nul (critère 2 de #13568)

`+raft` fait passer de **1/3** à **3/3** buts : $g_1$ (balise) et $g_2$ (trésor) sont **nouvellement atteignables**. C'est l'élargissement réel au niveau du **système** — exactement le différentiel de Planners-5c, désormais exprimé sur le quadruplet : $A$ croît (36 → 60 états) **et** les buts bougent.


## Contrôle négatif : la primitive stérile (critère 3)

Un différentiel *toujours positif* ne mesure pas, il décore. Le contrôle obligatoire : un ajout de primitive qui **n'élargit rien**. `discard-plank` (jeter la planchette) est le candidat — dans $A_t$ il n'y a rien à traverser, la planchette ne sert à rien, la détruire ne peut pas aider.


In [5]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Controle negatif par deltas : pour chaque vocabulaire etendu, quels buts de
// A_t gagne-t-on (nouveaux) et lesquels perd-on (perdus) ? C'est la lecture
// causale de la grille 2x2 — pas seulement des tailles d'espace, mais ce que
// chaque extension OU VRE et ne ferme jamais.

var vocabsB = Vocabulaires.Tous();
var butsB = Domaine.Buts();
var nomsButsB = butsB.Keys.OrderBy(k => k, StringComparer.Ordinal).ToArray();
Func<Dictionary<string, Etat>, string[]> butsAtteints = etats =>
    nomsButsB.Where(b => etats.Values.Any(s => s.Contient(butsB[b]))).ToArray();

var baseOk = butsAtteints(Banc.Atteint["A_t"]);
Console.WriteLine("Buts atteignables dans A_t : [" + string.Join(", ", baseOk.Select(b => "'" + b.Substring(0, 2) + "'")) + "]");

foreach (var v in vocabsB)
{
    if (v.Nom == "A_t") continue;
    var ok = butsAtteints(Banc.Atteint[v.Nom]);
    var nouveaux = ok.Where(b => !baseOk.Contains(b, StringComparer.Ordinal)).OrderBy(b => b, StringComparer.Ordinal).ToArray();
    var perdus = baseOk.Where(b => !ok.Contains(b, StringComparer.Ordinal)).OrderBy(b => b, StringComparer.Ordinal).ToArray();
    string nTxt = nouveaux.Length == 0 ? "AUCUN" : "[" + string.Join(", ", nouveaux.Select(b => "'" + b.Substring(0, 2) + "'")) + "]";
    string pTxt = perdus.Length == 0 ? "aucun" : "[" + string.Join(", ", perdus.Select(b => "'" + b.Substring(0, 2) + "'")) + "]";
    Console.WriteLine("A_t -> " + v.Nom.PadRight(16) + " nouveaux = " + nTxt + "   perdus = " + pTxt);
}

// Sterilite MESUREE du discard : aucun but nouveau, aucun but perdu —
// l'operateur en plus ne change pas ce qui est atteignable.
Verifs.Entier("buts de base dans A_t", baseOk.Length, 1);
Verifs.Entier("A_t+discard : nombre de buts nouveaux",
    butsAtteints(Banc.Atteint["A_t+discard"]).Count(b => !baseOk.Contains(b, StringComparer.Ordinal)), 0);
Verifs.Entier("A_t+discard : nombre de buts perdus",
    baseOk.Count(b => !butsAtteints(Banc.Atteint["A_t+discard"]).Contains(b, StringComparer.Ordinal)), 0);
// Le raft ouvre exactement les deux buts de la rive droite (g1 la balise,
// g2 le tresor), et ne ferme rien : enumerer les nouveaux par leur nom.
foreach (var nom in new[] { "A_t1 (+raft)", "A_t1+discard" })
{
    var ok = butsAtteints(Banc.Atteint[nom]);
    Verifs.Entier(nom + " : nombre de buts nouveaux",
        ok.Count(b => !baseOk.Contains(b, StringComparer.Ordinal)), 2);
    Verifs.Entier(nom + " : nombre de buts perdus",
        baseOk.Count(b => !ok.Contains(b, StringComparer.Ordinal)), 0);
    Verifs.Vrai(nom + " : les deux nouveaux buts sont g1 et g2",
        ok.Where(b => !baseOk.Contains(b, StringComparer.Ordinal)).OrderBy(b => b, StringComparer.Ordinal)
          .Select(b => b.Substring(0, 2)).SequenceEqual(new[] { "g1", "g2" }));
}

Buts atteignables dans A_t : ['g3']


A_t -> A_t+discard      nouveaux = AUCUN   perdus = aucun


A_t -> A_t1 (+raft)     nouveaux = ['g1', 'g2']   perdus = aucun


A_t -> A_t1+discard     nouveaux = ['g1', 'g2']   perdus = aucun


CONFORME — buts de base dans A_t : calcule = 1 | attendu = 1


CONFORME — A_t+discard : nombre de buts nouveaux : calcule = 0 | attendu = 0


... (3 outputs left out)

CONFORME — A_t1 (+raft) : les deux nouveaux buts sont g1 et g2


CONFORME — A_t1+discard : nombre de buts nouveaux : calcule = 2 | attendu = 2


CONFORME — A_t1+discard : nombre de buts perdus : calcule = 0 | attendu = 0


CONFORME — A_t1+discard : les deux nouveaux buts sont g1 et g2


### Lecture : $|\Delta A|$ décore, $\Delta(\text{buts})$ discrimine

Le résultat central du contrôle négatif : `A_t -> A_t+discard` fait croître $|A|$ de **36 à 54 états (+50 %)** sans rendre **aucun** but nouveau. Une mesure naïve de « progrès » par croissance d'espace ($|\Delta A| > 0$) validerait cet ajout — à tort. Le différentiel qui mesure est $\Delta(\text{buts atteignables})$, et lui reste **exactement nul**.

Verbatim de la sortie : « A_t -> A_t+discard nouveaux = AUCUN perdus = aucun » puis « A_t -> A_t1 (+raft) nouveaux = ['g1', 'g2'] perdus = aucun » — `discard` n'ajoute aucun but, `raft` en ajoute deux sans en perdre. La nuance est dans l'écart entre les deux colonnes : `discard` ne change pas les **buts**, mais il change bien $|A|$ (36 → 54 états) — l'opérateur ajoute des états, pas des objectifs atteignables.


La leçon pour la strate 7 : « élargir son espace » n'a de valeur que relativement à ce que l'agent **veut atteindre**. Un espace qui croît de 50 % sans ouvrir aucun but est du bruit, pas du progrès.


## $F$ et $r$ : le solveur a un budget

Le système n'est pas le solveur. Un BFS réel s'arrête à $B$ expansions — c'est le mur de Planners-10b (`BUDGET_NODES = 15 000`). Ici $F_B$ = découvert à budget $B$, $r = |F_B|/|A|$.


In [6]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// F_B (decouvert sous budget B) et r = |F_B| / |A|, pour les quatre vocabulaires.
var vocabsC = Vocabulaires.Tous();
var butsC = Domaine.Buts();
var nomsButsC = butsC.Keys.OrderBy(k => k, StringComparer.Ordinal).ToArray();
var fParBudget = new Dictionary<string, Dictionary<int, int>>(StringComparer.Ordinal);
var rParBudget = new Dictionary<string, Dictionary<int, double>>(StringComparer.Ordinal);
var butsFParBudget = new Dictionary<string, Dictionary<int, string[]>>(StringComparer.Ordinal);

foreach (int B in new[] { 30, 60, 120 })
{
    Console.WriteLine("budget B = " + B.ToString(CultureInfo.InvariantCulture) + " :");
    foreach (var v in vocabsC)
    {
        var etatsF = new Dictionary<string, Etat>(StringComparer.Ordinal);
        var distF = Strips.BfsBudget(Domaine.Initial(), v.Ops, B, etatsF);
        int tailleA = Banc.Dist[v.Nom].Count;
        double r = (double)distF.Count / tailleA;
        var butsOk = nomsButsC.Where(b => etatsF.Values.Any(s => s.Contient(butsC[b])))
            .Select(b => b.Substring(0, 2)).ToArray();

        Console.WriteLine("  " + v.Nom.PadRight(16)
            + " |F_B| = " + distF.Count.ToString(CultureInfo.InvariantCulture).PadLeft(4)
            + "   r = " + (r * 100.0).ToString("F1", CultureInfo.InvariantCulture) + " %"
            + "   buts dans F : [" + string.Join(", ", butsOk.Select(x => "'" + x + "'")) + "]");

        if (!fParBudget.ContainsKey(v.Nom))
        {
            fParBudget[v.Nom] = new Dictionary<int, int>();
            rParBudget[v.Nom] = new Dictionary<int, double>();
            butsFParBudget[v.Nom] = new Dictionary<int, string[]>();
        }
        fParBudget[v.Nom][B] = distF.Count;
        rParBudget[v.Nom][B] = r;
        butsFParBudget[v.Nom][B] = butsOk;
    }
}

// Controles calcules sur les valeurs reellement produites par les balayages.
foreach (var v in vocabsC)
{
    foreach (int B in new[] { 30, 60, 120 })
    {
        Verifs.Vrai(v.Nom + " a B=" + B.ToString(CultureInfo.InvariantCulture) + " : F_B est contenu dans A",
            fParBudget[v.Nom][B] <= Banc.Dist[v.Nom].Count);
        Verifs.Vrai(v.Nom + " a B=" + B.ToString(CultureInfo.InvariantCulture) + " : r est dans [0,1]",
            rParBudget[v.Nom][B] >= 0.0 && rParBudget[v.Nom][B] <= 1.0);
    }
    Verifs.Vrai(v.Nom + " : r est croissant avec le budget",
        rParBudget[v.Nom][30] <= rParBudget[v.Nom][60] && rParBudget[v.Nom][60] <= rParBudget[v.Nom][120]);
    Verifs.Entier(v.Nom + " a B=120 : |F_B| = |A| (r = 100 %)",
        fParBudget[v.Nom][120], Banc.Dist[v.Nom].Count);
}

// A B=30, l'elargissement contracte la liberte residuelle.
Verifs.Vrai("a B=30, r(A_t1+discard) < r(A_t1) < r(A_t) : l'elargissement contracte r",
    rParBudget["A_t1+discard"][30] < rParBudget["A_t1 (+raft)"][30]
    && rParBudget["A_t1 (+raft)"][30] < rParBudget["A_t"][30]);

// Le noyau du solveur : g1 quitte le front atteignable de F a budget serre.
Verifs.Vrai("a B=30, g1 est dans F(A_t1) mais hors de F(A_t1+discard) : le noyau du solveur",
    butsFParBudget["A_t1 (+raft)"][30].Contains("g1")
    && !butsFParBudget["A_t1+discard"][30].Contains("g1"));

// A B=120, chaque vocabulaire a bien explore tout son espace ambient.
// Ce controle ne confond pas exploration complete et atteignabilite de chaque but :
// A_t ne peut pas atteindre g1 et g2, meme lorsque tout A_t a ete decouvert.
foreach (var v in vocabsC)
{
    Verifs.Entier(v.Nom + " a B=120 : |F_B| = |A|",
        fParBudget[v.Nom][120], Banc.Dist[v.Nom].Count);
    Verifs.Vrai(v.Nom + " a B=120 : r vaut exactement 100 %",
        Math.Abs(rParBudget[v.Nom][120] - 1.0) < 1e-12);
}

// --- Parite exacte avec la source Python -------------------------------------
// References relevees sur le carnet source, pour B = 30, 60, 120 : |F_B|,
// |A| (pour recalculer r) et la liste des buts decouverts dans F.
var budgetsRef = new[] { 30, 60, 120 };
var attendusF = new Dictionary<string, int[]>(StringComparer.Ordinal)
{
    ["A_t"] = new[] { 34, 36, 36 },
    ["A_t+discard"] = new[] { 38, 54, 54 },
    ["A_t1 (+raft)"] = new[] { 40, 60, 60 },
    ["A_t1+discard"] = new[] { 42, 70, 78 },
};
var attendusARef = new Dictionary<string, int>(StringComparer.Ordinal)
{
    ["A_t"] = 36,
    ["A_t+discard"] = 54,
    ["A_t1 (+raft)"] = 60,
    ["A_t1+discard"] = 78,
};
var attendusButsF = new Dictionary<string, string[][]>(StringComparer.Ordinal)
{
    ["A_t"] = new[] { new[] { "g3" }, new[] { "g3" }, new[] { "g3" } },
    ["A_t+discard"] = new[] { new[] { "g3" }, new[] { "g3" }, new[] { "g3" } },
    ["A_t1 (+raft)"] = new[] { new[] { "g1", "g2", "g3" }, new[] { "g1", "g2", "g3" }, new[] { "g1", "g2", "g3" } },
    ["A_t1+discard"] = new[] { new[] { "g2", "g3" }, new[] { "g1", "g2", "g3" }, new[] { "g1", "g2", "g3" } },
};

foreach (var v in vocabsC)
{
    for (int k = 0; k < budgetsRef.Length; k++)
    {
        int B = budgetsRef[k];
        string Btxt = B.ToString(CultureInfo.InvariantCulture);
        Verifs.Entier(v.Nom + " a B=" + Btxt + " : |F_B| (parite source)",
            fParBudget[v.Nom][B], attendusF[v.Nom][k]);
        double rAttendu = (double)attendusF[v.Nom][k] / attendusARef[v.Nom];
        Verifs.Reel(v.Nom + " a B=" + Btxt + " : r = |F_B|/|A| (parite source)",
            rParBudget[v.Nom][B], rAttendu, 1e-9);
        Verifs.Reel(v.Nom + " a B=" + Btxt + " : r en % (parite source)",
            Math.Round(rParBudget[v.Nom][B] * 100.0, 1), Math.Round(rAttendu * 100.0, 1), 1e-9);
        Verifs.Exact(v.Nom + " a B=" + Btxt + " : buts dans F (parite source)",
            string.Join(",", butsFParBudget[v.Nom][B]), string.Join(",", attendusButsF[v.Nom][k]));
    }
}


budget B = 30 :


  A_t              |F_B| =   34   r = 94.4 %   buts dans F : ['g3']


  A_t+discard      |F_B| =   38   r = 70.4 %   buts dans F : ['g3']


  A_t1 (+raft)     |F_B| =   40   r = 66.7 %   buts dans F : ['g1', 'g2', 'g3']


  A_t1+discard     |F_B| =   42   r = 53.8 %   buts dans F : ['g2', 'g3']


budget B = 60 :


... (95 outputs left out)

CONFORME — A_t1+discard a B=120 : |F_B| (parite source) : calcule = 78 | attendu = 78


CONFORME — A_t1+discard a B=120 : r = |F_B|/|A| (parite source) : calcule = 1 | attendu = 1


CONFORME — A_t1+discard a B=120 : r en % (parite source) : calcule = 100 | attendu = 100


CONFORME — A_t1+discard a B=120 : buts dans F (parite source) : calcule = g1,g2,g3 | attendu = g1,g2,g3


### Lecture : l'élargissement non contrôlé **contracte** la liberté résiduelle

À budget $B=30$ : $r$ chute de **94,4 %** ($A_t$) à **53,8 %** ($A_{t1}$+discard). Le prix se paie deux fois : $r$ descend de **94,4 %** ($A_t$) à **66,7 %** ($A_{t1}$), puis à **53,8 %** ($A_{t1}$+discard). Le tableau réserve un piège : $A_{t1}$+discard affiche « buts dans F : ['g2', 'g3'] » — à budget serré, `discard` fait **perdre $g_1$** du front atteignable, alors même qu'il n'ajoutait aucun but. À $B=120$, $r$ remonte à **100,0 %** partout : la contraction n'est pas une propriété du système, c'est une propriété du **budget**.


Le quadruplet isole le paradoxe : élargir $A$ (le système peut plus) peut **réduire** $r$ (le solveur explore une part plus petite de ce qui existe). L'espace croît plus vite que le budget — c'est la face sombre de l'élargissement, invisible si on ne regarde que $A$.


## Le noyau : ce que la primitive ajoutée rend **inatteignable** (critère 4)

L'obligation méthodologique de #13565/#13568 : ne jamais conclure « élargissement » sans chercher le **noyau** — ce que l'ajout rend inatteignable. Deux niveaux à distinguer honnêtement :

1. **Noyau système** : dans STRIPS, les actions sont *optionnelles* — un ancien plan reste valide après l'ajout (monotonie de l'atteignabilité). Le noyau système est donc **vide** par construction : la mesure ci-dessus le confirme (`perdus = aucun` partout).
2. **Noyau à budget fixe** : le solveur, lui, n'a pas cette clémence. À $B$ constant, élargir le vocabulaire dilue le budget sur plus de successeurs — des buts qui tiennent dans $F$ peuvent en sortir.


In [7]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

public static class Noyau
{
    // BFS interrompue a la PREMIERE decouverte du but : expansions compte les
    // etats depiles avant elle (le cout de la recherche aveugle), distance est
    // la profondeur du but — c'est h* pour ce but dans ce vocabulaire, car une
    // BFS decouvre chaque etat a sa profondeur minimale. Un espace epuise sans
    // but renvoie (null, null) : le but est hors de portee, pas « lent ».
    public static ResultatBut ExpansionsPourBut(
        Etat initial,
        IEnumerable<Operateur> vocabulaire,
        IEnumerable<string> but,
        Func<Operateur, CleTri> cleTri = null)
    {
        cleTri ??= Strips.CleLex;
        var condition = but.ToArray();
        var dist = new Dictionary<string, int>(StringComparer.Ordinal) { [initial.Key] = 0 };
        var etats = new Dictionary<string, Etat>(StringComparer.Ordinal) { [initial.Key] = initial };
        var file = new Queue<Etat>();
        file.Enqueue(initial);
        int expansions = 0;
        while (file.Count > 0)
        {
            var s = file.Dequeue();
            expansions++;
            if (s.Contient(condition)) return new ResultatBut(expansions, dist[s.Key]);
            foreach (var (_, t) in Strips.Successeurs(s, vocabulaire, cleTri))
                if (!dist.ContainsKey(t.Key))
                {
                    dist[t.Key] = dist[s.Key] + 1;
                    etats[t.Key] = t;
                    file.Enqueue(t);
                }
        }
        return new ResultatBut(null, null);
    }
}

var vocabsD = Vocabulaires.Tous();
var butsD = Domaine.Buts();
var nomsButsD = butsD.Keys.OrderBy(k => k, StringComparer.Ordinal).ToArray();
var resNoyau = new Dictionary<string, Dictionary<string, ResultatBut>>(StringComparer.Ordinal);

Console.WriteLine("Expansions necessaires a la premiere decouverte de chaque but :");
foreach (var bNom in nomsButsD)
{
    var ligne = new Dictionary<string, ResultatBut>(StringComparer.Ordinal);
    foreach (var v in vocabsD)
    {
        var resultat = Noyau.ExpansionsPourBut(Domaine.Initial(), v.Ops, butsD[bNom]);
        ligne[v.Nom] = resultat;
    }
    resNoyau[bNom] = ligne;
    Console.WriteLine("  " + bNom.PadRight(24) + string.Join("  ",
        vocabsD.Select(v => v.Nom + "=" + ligne[v.Nom].ExpansionsTexte)));
}

// Attendus ancrees de la source : None = hors de portee (confinement), le
// chiffre = expansions jusqu'a la premiere decouverte du but.
var attendusNoyau = new Dictionary<string, int?[]>(StringComparer.Ordinal)
{
    ["g1: balise (5,2)"] = new int?[] { null, null, 39, 50 },
    ["g2: detenir le tresor"] = new int?[] { null, null, 33, 42 },
    ["g3: detenir la cle"] = new int?[] { 8, 8, 8, 8 },
};
foreach (var bNom in nomsButsD)
{
    for (int i = 0; i < vocabsD.Count; i++)
    {
        var calcule = resNoyau[bNom][vocabsD[i].Nom].Expansions;
        var attendu = attendusNoyau[bNom][i];
        bool ok = calcule.HasValue == attendu.HasValue && (!calcule.HasValue || calcule.Value == attendu.Value);
        Console.WriteLine((ok ? "CONFORME" : "ECART") + " — expansions de " + bNom + " dans " + vocabsD[i].Nom
            + " : calcule = " + (calcule.HasValue ? calcule.Value.ToString(CultureInfo.InvariantCulture) : "None")
            + " | attendu = " + (attendu.HasValue ? attendu.Value.ToString(CultureInfo.InvariantCulture) : "None"));
        if (!ok) throw new InvalidOperationException("Ecart sur le controle des expansions de " + bNom);
    }
}

// Confinement mesurer : sans le raft, g1 et g2 ne sont pas « difficiles »,
// ils sont INATTEIGNABLES — la BFS epuise l'espace sans jamais les satisfaire.
Verifs.Vrai("g1 est hors de portee sans le raft",
    !resNoyau["g1: balise (5,2)"]["A_t"].Expansions.HasValue
    && !resNoyau["g1: balise (5,2)"]["A_t+discard"].Expansions.HasValue);
Verifs.Vrai("g2 est hors de portee sans le raft",
    !resNoyau["g2: detenir le tresor"]["A_t"].Expansions.HasValue
    && !resNoyau["g2: detenir le tresor"]["A_t+discard"].Expansions.HasValue);
// g3 (la cle, rive gauche) est immunise au contexte : meme cout partout.
Verifs.Vrai("g3 est atteint partout en 8 expansions",
    vocabsD.All(v => resNoyau["g3: detenir la cle"][v.Nom].Expansions == 8));
// Le cout du bruit : discard ne change pas la profondeur optimale (h* fixe)
// mais rallonge la decouverte — la primitive sterile ne fait qu'encombrer
// la file d'attente de la recherche.
Verifs.Vrai("discard allonge la decouverte de g1",
    resNoyau["g1: balise (5,2)"]["A_t1 (+raft)"].Expansions < resNoyau["g1: balise (5,2)"]["A_t1+discard"].Expansions);
Verifs.Vrai("discard allonge la decouverte de g2",
    resNoyau["g2: detenir le tresor"]["A_t1 (+raft)"].Expansions < resNoyau["g2: detenir le tresor"]["A_t1+discard"].Expansions);
Verifs.Vrai("h* est inchange par l'ajout de discard",
    resNoyau["g1: balise (5,2)"]["A_t1 (+raft)"].Distance == resNoyau["g1: balise (5,2)"]["A_t1+discard"].Distance
    && resNoyau["g2: detenir le tresor"]["A_t1 (+raft)"].Distance == resNoyau["g2: detenir le tresor"]["A_t1+discard"].Distance);

Expansions necessaires a la premiere decouverte de chaque but :


  g1: balise (5,2)        A_t=None  A_t+discard=None  A_t1 (+raft)=39  A_t1+discard=50


  g2: detenir le tresor   A_t=None  A_t+discard=None  A_t1 (+raft)=33  A_t1+discard=42


  g3: detenir la cle      A_t=8  A_t+discard=8  A_t1 (+raft)=8  A_t1+discard=8


CONFORME — expansions de g1: balise (5,2) dans A_t : calcule = None | attendu = None


CONFORME — expansions de g1: balise (5,2) dans A_t+discard : calcule = None | attendu = None


... (12 outputs left out)

CONFORME — g3 est atteint partout en 8 expansions


CONFORME — discard allonge la decouverte de g1


CONFORME — discard allonge la decouverte de g2


CONFORME — h* est inchange par l'ajout de discard


### Lecture : le noyau non vide vit à budget fixe

Verbatim des expansions à la première découverte : $g_1$ « A_t=None A_t+discard=None A_t1 (+raft)=39 A_t1+discard=50 », $g_2$ « None None 33 42 », $g_3$ « 8 8 8 8 » — $g_1$ et $g_2$ exigent le raft (indisponible → `None`), $g_3$ est atteint partout en 8 expansions. La nécessité du raft se lit sur deux lignes, l'indifférence sur la troisième.


- $g_1$ : **inatteignable** dans $A_t$ et $A_t$+discard (le gouffre), découvert à **39** expansions dans $A_{t1}$ — mais à **50** dans $A_{t1}$+discard ;
- $g_2$ : **33** dans $A_{t1}$ contre **42** dans $A_{t1}$+discard.

Lecture directe sur le tableau $F/r$ à $B=30$ : $g_1$ est **dans** $F(A_{t1})$ mais **hors** $F(A_{t1}$+discard$)$ — l'ajout de `discard` (une primitive qui ne sert à *rien*) a rendu $g_1$ inatteignable **au solveur à budget constant**. C'est le noyau : pas au système (monotonie), mais au solveur (dilution). Les deux niveaux sont mesurés et distingués — c'est exactement ce que le critère 4 exige.


## Cas 3 — navigation : seul $\pi$ bouge (le bras Planners-10b)

Troisième signature : $A$ et $F$ **inchangés**, seule la **politique** change. C'est la position de [Planners-10b](../../SymbolicAI/Planners/04-NeuroSymbolic/Planners-10b-LLM-Space-Reducer-Python.ipynb) : le LLM ne résout pas, il **réduit** l'espace exploré en premier (quelles primitives essayer d'abord) ; le solving reste déterministe. Ici, la version jouet : réordonner l'expansion — `pickup` d'abord, ou `raft` d'abord, contre l'ordre lexicographique.


In [8]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

Func<Operateur, CleTri> piLex = o => new CleTri(0, o.Nom);
Func<Operateur, CleTri> piPickup = o => new CleTri(o.Nom.StartsWith("pickup", StringComparison.Ordinal) ? 0 : 1, o.Nom);
Func<Operateur, CleTri> piRaft = o => new CleTri(o.Nom.StartsWith("raft", StringComparison.Ordinal) ? 0 : 1, o.Nom);

var aT1 = Vocabulaires.Tous().First(v => v.Nom == "A_t1 (+raft)").Ops;
var butsE = Domaine.Buts();
var nomsButsE = butsE.Keys.OrderBy(k => k, StringComparer.Ordinal).ToArray();
var resPi = new Dictionary<string, Dictionary<string, ResultatBut>>(StringComparer.Ordinal);

Console.WriteLine("A_t1, trois politiques -- A et F inchanges par construction (meme dynamique) :");
Console.WriteLine("but".PadRight(24) + "pi_lex".PadLeft(8) + "pi_pickup".PadLeft(10) + "pi_raft".PadLeft(8) + "   h*");
foreach (var bNom in nomsButsE)
{
    var r0 = Noyau.ExpansionsPourBut(Domaine.Initial(), aT1, butsE[bNom], piLex);
    var r1 = Noyau.ExpansionsPourBut(Domaine.Initial(), aT1, butsE[bNom], piPickup);
    var r2 = Noyau.ExpansionsPourBut(Domaine.Initial(), aT1, butsE[bNom], piRaft);
    resPi[bNom] = new Dictionary<string, ResultatBut>(StringComparer.Ordinal)
    {
        ["pi_lex"] = r0,
        ["pi_pickup"] = r1,
        ["pi_raft"] = r2,
    };
    Console.WriteLine("  " + bNom.PadRight(22) + r0.ExpansionsTexte.PadLeft(7)
        + r1.ExpansionsTexte.PadLeft(9) + r2.ExpansionsTexte.PadLeft(7)
        + "   " + r0.DistanceTexte + " (inchange)");
}

Verifs.Entier("pi_lex : expansions pour g2", resPi["g2: detenir le tresor"]["pi_lex"].Expansions ?? -1, 33);
Verifs.Entier("pi_pickup : expansions pour g2", resPi["g2: detenir le tresor"]["pi_pickup"].Expansions ?? -1, 32);
Verifs.Entier("pi_lex : expansions pour g3", resPi["g3: detenir la cle"]["pi_lex"].Expansions ?? -1, 8);
Verifs.Entier("pi_pickup : expansions pour g3", resPi["g3: detenir la cle"]["pi_pickup"].Expansions ?? -1, 7);
foreach (var bNom in nomsButsE)
{
    var h = resPi[bNom]["pi_lex"].Distance;
    Verifs.Vrai("h* inchange sur " + bNom,
        resPi[bNom]["pi_pickup"].Distance == h && resPi[bNom]["pi_raft"].Distance == h);
}
Verifs.Vrai("pi_pickup ameliore g2 et g3",
    resPi["g2: detenir le tresor"]["pi_pickup"].Expansions < resPi["g2: detenir le tresor"]["pi_lex"].Expansions
    && resPi["g3: detenir la cle"]["pi_pickup"].Expansions < resPi["g3: detenir la cle"]["pi_lex"].Expansions);

// Parite exacte avec la source : ligne g1 (39/39/39), colonne pi_raft,
// et les valeurs exactes de h* (7, 6, 3) pour les trois politiques.
Verifs.Entier("pi_lex : expansions pour g1", resPi["g1: balise (5,2)"]["pi_lex"].Expansions ?? -1, 39);
Verifs.Entier("pi_pickup : expansions pour g1", resPi["g1: balise (5,2)"]["pi_pickup"].Expansions ?? -1, 39);
Verifs.Entier("pi_raft : expansions pour g1", resPi["g1: balise (5,2)"]["pi_raft"].Expansions ?? -1, 39);
Verifs.Entier("pi_raft : expansions pour g2", resPi["g2: detenir le tresor"]["pi_raft"].Expansions ?? -1, 32);
Verifs.Entier("pi_raft : expansions pour g3", resPi["g3: detenir la cle"]["pi_raft"].Expansions ?? -1, 8);
Verifs.Entier("h* pour g1 (pi_lex)", resPi["g1: balise (5,2)"]["pi_lex"].Distance ?? -1, 7);
Verifs.Entier("h* pour g2 (pi_lex)", resPi["g2: detenir le tresor"]["pi_lex"].Distance ?? -1, 6);
Verifs.Entier("h* pour g3 (pi_lex)", resPi["g3: detenir la cle"]["pi_lex"].Distance ?? -1, 3);
Verifs.Entier("h* pour g1 (pi_pickup)", resPi["g1: balise (5,2)"]["pi_pickup"].Distance ?? -1, 7);
Verifs.Entier("h* pour g2 (pi_pickup)", resPi["g2: detenir le tresor"]["pi_pickup"].Distance ?? -1, 6);
Verifs.Entier("h* pour g3 (pi_pickup)", resPi["g3: detenir la cle"]["pi_pickup"].Distance ?? -1, 3);
Verifs.Entier("h* pour g1 (pi_raft)", resPi["g1: balise (5,2)"]["pi_raft"].Distance ?? -1, 7);
Verifs.Entier("h* pour g2 (pi_raft)", resPi["g2: detenir le tresor"]["pi_raft"].Distance ?? -1, 6);
Verifs.Entier("h* pour g3 (pi_raft)", resPi["g3: detenir la cle"]["pi_raft"].Distance ?? -1, 3);


A_t1, trois politiques -- A et F inchanges par construction (meme dynamique) :


but                       pi_lex pi_pickup pi_raft   h*


  g1: balise (5,2)           39       39     39   7 (inchange)


  g2: detenir le tresor      33       32     32   6 (inchange)


  g3: detenir la cle          8        7      8   3 (inchange)


CONFORME — pi_lex : expansions pour g2 : calcule = 33 | attendu = 33


... (17 outputs left out)

CONFORME — h* pour g3 (pi_pickup) : calcule = 3 | attendu = 3


CONFORME — h* pour g1 (pi_raft) : calcule = 7 | attendu = 7


CONFORME — h* pour g2 (pi_raft) : calcule = 6 | attendu = 6


CONFORME — h* pour g3 (pi_raft) : calcule = 3 | attendu = 3


### Lecture : trois signatures, un seul quadruplet

$\pi_{pickup}$ améliore la découverte de $g_2$ (33 → 32) et $g_3$ (8 → 7) ; $h^*$ est **inchangé** partout (la distance optimale ne dépend pas de l'ordre d'exploration — garanti par le lake : l'admissibilité de $h^+$ ne connaît pas $\pi$). La table des trois cas de la strate 7, sur le même domaine :

| Cas | $A$ | $F_B$ | $r$ | Témoin mesuré |
|---|---|---|---|---|
| **Élargissement réel** | croît (36→60) | croît | varie | `+raft` : 1/3 → 3/3 buts |
| **Dilution (face sombre)** | croît (60→78) | stagne à budget | **chute** (66,7 %→53,8 %) | `+discard` : $g_1$ sort de $F$ à $B{=}30$ |
| **Navigation** | **inchangé** | **inchangé** | inchangé | $\pi_{pickup}$ : exp($g_3$) 8 → 7, $h^*$ stable |

« L'agent a progressé » ne dit rien tant que la ligne du tableau n'est pas nommée. C'est la réponse du quadruplet à la formule nébuleuse.


## À retenir

*Planners-5c mesurait le différentiel d'atteignabilité mais n'avait pas le contrôle négatif **exécuté** (exercice 2 non résolu), ni le noyau, ni $r$. Planners-10b montrait la réduction LLM sur un autre domaine (grilles ARC-like), sans lien avec l'élargissement. La greffe :*

1. **nomme** le quadruplet $(A, F, r, \pi)$ — les quatre objets, mesurables séparément ;
2. **exécute** le contrôle négatif : +50 % d'états, 0 but nouveau — la croissance d'espace n'est pas le progrès ;
3. **mesure le noyau** à deux niveaux : vide au système (monotonie STRIPS, confirmée), non vide au solveur à budget ;
4. **discrimine les trois cas** de la strate 7 sur le même banc — le témoin de contraction *certifiée* (#13565, Beck-Fiala) viendra greffer le quatrième quand livré.


## Limites

- Le domaine est **petit** (36-78 états) : les effets de budget sont visibles à $B \in [30, 120]$, pas à l'échelle du mur de Planners-10b (15 000 nœuds). La structure des trois signatures est identique ; les magnitudes ne se transfèrent pas linéairement.
- La monotonie STRIPS (noyau système vide) est une propriété du **cadre** (actions optionnelles, sans contraintes globales) : un domaine à invariants (capacité, Beck-Fiala) peut avoir un noyau système non vide — c'est l'exercice 3.
- $\pi$ jouet : trois ordres statiques. Le réducteur de Planners-10b est *par tâche* (le LLM propose le sous-ensemble pertinent) — une politique dépendante de l'état, plus riche que nos clés de tri constantes.
- Les proxys $|\Delta A|$, $r$ sont des **bornes d'exploration**, pas des prédicteurs de performance d'agent complet (pas de récompense, pas d'apprentissage ici).


## Vérifiez votre compréhension

1. Le contrôle négatif exécuté donne « +50 % d'états, 0 but nouveau » : que prouve-t-il ?
2. Le noyau est mesuré à deux niveaux — vide au système, non vide au solveur à budget : pourquoi les distinguer ?
3. La monotonie STRIPS est dite « une propriété du cadre » : à quelle condition le noyau système devient-il non vide ?

<details>
<summary>Réponses</summary>

1. Que **la croissance d'espace n'est pas le progrès** : |ΔA| est une borne d'exploration, et un élargissement qui n'ouvre aucun but atteignable nouveau est un ajout inutile. L'exercice 1 pousse la logique un cran plus loin : avec `burn-plank`, l'état-piège rend l'ajout **nuisible** — inutile et nuisible sont deux échecs distincts de la greffe.
2. La monotonie STRIPS garantit qu'aucun ajout ne rend un but inatteignable — le noyau **système** est vide par propriété du cadre, et le carnet le confirme. Mais le solveur à budget ($B \in [30, 120]$) peut ne pas trouver dans son quota : cette contraction **apparente** est un artefact de budget, pas de sémantique. Sans les deux niveaux, on lirait une limite de calcul comme une perte réelle d'atteignabilité.
3. Quand le domaine porte des **invariants** : la monotonie tient aux actions optionnelles sans contraintes globales ; un domaine à contrainte de capacité (l'invariant carry-capacity-1 / main-libre de l'exercice 3, ou Beck-Fiala) peut rendre un but atteignable inatteignable après greffe — le noyau système devient non vide. C'est le quatrième cas que le témoin de contraction *certifiée* (#13565) viendra greffer sur ce banc.

</details>


In [9]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Exercice 1 a completer : la primitive qui piege reellement.
// TODO etudiant : ajouter burn-plank (detruire la planchette AVANT pickup : etat-piege plus precoce
// que discard -- preconditions {plank-at-x1-y1} sans holding), construire A_t1+burn,
// mesurer : (a) |A| croit-il davantage qu'avec discard ? (b) exp(g1) et exp(g2) :
// le noyau solveur croit-il ? (c) r a B=30 : conclusion sur la difference
// entre "ajout inutile" et "ajout nuisible".

// TODO etudiant : ecrire la construction du vocabulaire A_t1+burn (Operateur "burn-plank"),
// puis les trois mesures. Le moteur est deja en place :
//   - Strips.BfsBudget(Domaine.Initial(), vocabulaire, budget) pour |A| et |F_B| ;
//   - Noyau.ExpansionsPourBut(Domaine.Initial(), vocabulaire, conditionDuBut, cleTri) pour exp(g1)/exp(g2) ;
//   - Domaine.Buts() pour les conditions des trois buts.

// Valeurs neutres tant que l'exercice n'est pas fait (aucune valeur attendue fournie).
int? tailleA = null;         // (a) |A| du vocabulaire A_t1+burn
int? tailleADiscard = null;  // (a) |A| du vocabulaire A_t1+discard, pour comparaison
int? expG1 = null;           // (b) expansions a la premiere decouverte de g1
int? expG2 = null;           // (b) expansions a la premiere decouverte de g2
double? rBudget30 = null;    // (c) liberte residuelle a B=30
string? conclusion = null;   // (c) ajout inutile vs ajout nuisible

string Aff(int? v) => v.HasValue ? v.Value.ToString(CultureInfo.InvariantCulture) : "non mesure";

Console.WriteLine("Exercice a completer");
Console.WriteLine("  (a) |A|(A_t1+burn) = " + Aff(tailleA) + " | |A|(A_t1+discard) = " + Aff(tailleADiscard));
Console.WriteLine("  (b) exp(g1) = " + Aff(expG1) + " | exp(g2) = " + Aff(expG2));
Console.WriteLine("  (c) r(B=30) = " + (rBudget30.HasValue ? rBudget30.Value.ToString("F1", CultureInfo.InvariantCulture) : "non mesure")
    + " | conclusion = " + (conclusion ?? "non mesure"));


Exercice a completer


  (a) |A|(A_t1+burn) = non mesure | |A|(A_t1+discard) = non mesure


  (b) exp(g1) = non mesure | exp(g2) = non mesure


  (c) r(B=30) = non mesure | conclusion = non mesure


### Exercice 2 — La politique dégradante

Construire $\pi_{degrade}$ qui fait **pire** que l'ordre lexicographique sur au moins un but (indice : mettre les `move` inutiles en tête, ou trier par nom décroissant). Mesurer la dégradation, et vérifier que $h^*$ reste inchangé — la politique ne peut dégrader la *découverte*, jamais la *distance*.


In [10]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Exercice 2 a completer : pi_degrade = ... ; mesurer
// TODO etudiant : ecrire ici la politique degradee et les trois mesures.
// TODO etudiant : comparer a piLex et confirmer que h* reste inchange.

Func<Operateur, CleTri>? piDegrade = null;
int? expG1Lex = null;
int? expG1Deg = null;
int? expG2Lex = null;
int? expG2Deg = null;
int? expG3Lex = null;
int? expG3Deg = null;
int? hG1 = null;
int? hG2 = null;
int? hG3 = null;

string Aff(int? valeur) => valeur.HasValue ? valeur.Value.ToString(CultureInfo.InvariantCulture) : "non mesure";

Console.WriteLine("Exercice a completer");
Console.WriteLine("  pi_degrade definie = " + (piDegrade != null ? "oui" : "non"));
Console.WriteLine("  g1 : exp(pi_lex) = " + Aff(expG1Lex) + " | exp(pi_degrade) = " + Aff(expG1Deg) + " | h* = " + Aff(hG1));
Console.WriteLine("  g2 : exp(pi_lex) = " + Aff(expG2Lex) + " | exp(pi_degrade) = " + Aff(expG2Deg) + " | h* = " + Aff(hG2));
Console.WriteLine("  g3 : exp(pi_lex) = " + Aff(expG3Lex) + " | exp(pi_degrade) = " + Aff(expG3Deg) + " | h* = " + Aff(hG3));


Exercice a completer


  pi_degrade definie = non


  g1 : exp(pi_lex) = non mesure | exp(pi_degrade) = non mesure | h* = non mesure


  g2 : exp(pi_lex) = non mesure | exp(pi_degrade) = non mesure | h* = non mesure


  g3 : exp(pi_lex) = non mesure | exp(pi_degrade) = non mesure | h* = non mesure


### Exercice 3 — Le noyau **système** non vide : la contrainte d'invariant

Le noyau système est vide en STRIPS pur (actions optionnelles). Le rendre non vide : ajouter un **invariant global** — par exemple `carry-capacity-1` (on ne peut tenir qu'un objet à la fois : chaque `pickup-x` a en précondition `hand-free`, chaque `pickup-y` supprime `hand-free`). Construire ce vocabulaire $A_{t1}^{cap}$ et montrer qu'un but **atteignable dans $A_{t1}$** devient inatteignable : le noyau système, cette fois, est mesuré non vide. C'est le pont vers la contraction **certifiée** de #13565 (Beck-Fiala).


In [11]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Exercice 3 a completer : construire A_t1_cap (invariant hand-free sur les pickups),
// mesurer les buts atteignables, identifier le but perdu vs A_t1 -- le noyau systeme.

// TODO etudiant : ecrire la construction du vocabulaire contraint A_t1_cap : chaque pickup
// exige l'atome "hand-free" en precondition et le supprime de ses effets, puis comparer
// les buts atteignables de A_t1_cap a ceux de A_t1 (Moteur : Strips.BfsBudget(Domaine.Initial(),
// vocabulaire) pour l'espace atteint, Noyau.ExpansionsPourBut(...) pour les expansions).

// TODO etudiant : afficher le but perdu (celui qui est atteint dans A_t1 et plus dans A_t1_cap).
string? butPerdu = null;      // libelle du but devenu inatteignable dans A_t1_cap
int? tailleCap = null;       // |A| du vocabulaire A_t1_cap
int? tailleA_T1 = null;      // |A| du vocabulaire A_t1 (reference)
int? expButPerdu = null;     // expansions a la decouverte du but perdu dans A_t1 (reference)

string Aff(int? v) => v.HasValue ? v.Value.ToString(CultureInfo.InvariantCulture) : "non mesure";

Console.WriteLine("Exercice a completer");
Console.WriteLine("  |A|(A_t1) = " + Aff(tailleA_T1) + " | |A|(A_t1_cap) = " + Aff(tailleCap));
Console.WriteLine("  but perdu dans A_t1_cap = " + (butPerdu ?? "non mesure")
    + " | exp dans A_t1 = " + Aff(expButPerdu));


Exercice a completer


  |A|(A_t1) = non mesure | |A|(A_t1_cap) = non mesure


  but perdu dans A_t1_cap = non mesure | exp dans A_t1 = non mesure


## Pour aller plus loin

- Issue #13568 — la greffe 2 : quadruplet $(A, F, r, \pi)$, contrôle négatif, noyau.
- [Planners-5c — Le différentiel d'atteignabilité](../../SymbolicAI/Planners/02-Classical/Planners-05c-Differentiel-Atteignabilite-Python.ipynb) — le socle STRIPS et le domaine, copiés fidèlement.
- [Planners-10b — Le LLM comme réducteur d'espace](../../SymbolicAI/Planners/04-NeuroSymbolic/Planners-10b-LLM-Space-Reducer-Python.ipynb) — le bras restriction, moteur [aicpp](https://github.com/Julien-Livet/aicpp) (Julien Livet, Apache 2.0).
- #13565 — discrepancy_lean / Beck-Fiala : le témoin de contraction certifiée (à greffer).
- Lake `planning_lean` : `Relaxation.lean:56`, `Strips.lean:58`, `Admissibility.lean:41` — les trois garanties citées.
- **[ICT-Greffe4 — Le vote argumenté sur chaîne](ICT-Greffe4-VoteOnChain-Python.ipynb)** : fermer la boucle argumentation, choix social et exécution on-chain.
- **[ICT-0 — Cadrage de la série](ICT-0-Framing.md)** : le principe directeur de la série ICT.
